In [1]:

from pathlib import Path
import sys
import json

# Locate the project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Schema file exists:", (PROJECT_ROOT / "src" / "schema.py").exists())
print("Tools file exists:", (PROJECT_ROOT / "src" / "tools.py").exists())
print("Model file exists:", (PROJECT_ROOT / "src" / "model.py").exists())
print("Pipeline file exists:", (PROJECT_ROOT / "src" / "pipeline.py").exists())
print("Evaluation data exists:", (PROJECT_ROOT / "data" / "eval_set.jsonl").exists())


Project root: c:\Users\HP\AppliedGenAIAssignments
Schema file exists: True
Tools file exists: True
Model file exists: True
Pipeline file exists: True
Evaluation data exists: True


In [2]:

from src.schema import validate_output, OutputValidationError

context = "The Yuan dynasty was established by Kublai Khan in 1271."

# Test 1: A valid answer
valid_answer = {
    "answerable": True,
    "answer": "Kublai Khan",
    "evidence": context,
    "confidence": 0.95
}

result = validate_output(valid_answer, context)
print("Valid answer test: PASSED")
print(result)

# Test 2: A valid abstention
valid_abstention = {
    "answerable": False,
    "answer": "",
    "evidence": "",
    "confidence": 0.90
}

result = validate_output(valid_abstention, context)
print("\nAbstention test: PASSED")
print(result)


Valid answer test: PASSED
{'answerable': True, 'answer': 'Kublai Khan', 'evidence': 'The Yuan dynasty was established by Kublai Khan in 1271.', 'confidence': 0.95}

Abstention test: PASSED
{'answerable': False, 'answer': '', 'evidence': '', 'confidence': 0.9}


In [3]:

from src.tools import ContextRetriever

data_path = PROJECT_ROOT / "data" / "eval_set.jsonl"

retriever = ContextRetriever(data_path, top_k=3)

question = "Who established the Yuan dynasty?"
results = retriever.retrieve(question)

print("Unique contexts indexed:", len(retriever.contexts))
print("Retrieved passages:", len(results))

for i, result in enumerate(results, start=1):
    print(f"\nPassage {i}")
    print("Similarity:", round(result["score"], 4))
    print("Context:", result["context"][:500])


Unique contexts indexed: 35
Retrieved passages: 3

Passage 1
Similarity: 0.4939
Context: The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China established by Kublai Khan, leader of the Mongolian Borjigin clan. Although the Mongols had ruled territories including today's North China for decades, it was not until 1271 that Kublai Khan officially proclaimed the dynasty in the traditional Chinese style. His realm was, by this point, isolated from the other khana

Passage 2
Similarity: 0.0327
Context: Starting in the late 1950s, American computer scientist Paul Baran developed the concept Distributed Adaptive Message Block Switching with the goal to provide a fault-tolerant, efficient routing method for telecommunication messages as part of a research program at the RAND Corporation, funded by the US Department of Defense. This concept contrasted and contradicted th

In [4]:

test_questions = [
    "Who established the Yuan dynasty?",
    "What is the capital of Mars?"
]

for question in test_questions:
    results = retriever.retrieve(question)

    print(f"\nQuestion: {question}")

    if results:
        print("Top similarity:", round(results[0]["score"], 4))
        print("Top passage:", results[0]["context"][:200])
    else:
        print("No relevant passages found.")



Question: Who established the Yuan dynasty?
Top similarity: 0.4939
Top passage: The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China established by Kubl

Question: What is the capital of Mars?
Top similarity: 0.1721
Top passage: Warsaw (Polish: Warszawa [varˈʂava] ( listen); see also other names) is the capital and largest city of Poland. It stands on the Vistula River in east-central Poland, roughly 260 kilometres (160 mi) f


In [5]:

# Experiment with a minimum similarity threshold.
SIMILARITY_THRESHOLD = 0.20

for question in [
    "Who established the Yuan dynasty?",
    "What is the capital of Mars?"
]:
    results = retriever.retrieve(question)

    print(f"\nQuestion: {question}")

    if results and results[0]["score"] >= SIMILARITY_THRESHOLD:
        print("Status: Relevant context candidate")
        print("Similarity:", round(results[0]["score"], 4))
        print("Context:", results[0]["context"][:200])
    else:
        print("Status: No sufficiently relevant context")



Question: Who established the Yuan dynasty?
Status: Relevant context candidate
Similarity: 0.4939
Context: The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China established by Kubl

Question: What is the capital of Mars?
Status: No sufficiently relevant context


In [6]:

# Deliberately provide evidence that is not in the context.
invalid_answer = {
    "answerable": True,
    "answer": "Genghis Khan",
    "evidence": "Genghis Khan established the Yuan dynasty in 1271.",
    "confidence": 0.95
}

try:
    validate_output(invalid_answer, context)
    print("ERROR: Invalid evidence was accepted.")
except OutputValidationError as exc:
    print("Invalid evidence test: PASSED")
    print("Rejection reason:", exc)


Invalid evidence test: PASSED
Rejection reason: Evidence is not an exact passage from the context.


In [7]:

threshold = 0.20

test_questions = [
    "Who established the Yuan dynasty?",
    "When was the Yuan dynasty proclaimed?",
    "What is the capital of Mars?",
    "What is the University of Chicago?"
]

for question in test_questions:
    results = retriever.retrieve(question)
    score = results[0]["score"] if results else 0.0
    accepted = score >= threshold

    print(f"\nQuestion: {question}")
    print(f"Top similarity: {score:.4f}")
    print(f"Threshold decision: {'ACCEPT' if accepted else 'REJECT'}")
    if accepted:
        print("Candidate context:", results[0]["context"][:180])



Question: Who established the Yuan dynasty?
Top similarity: 0.4939
Threshold decision: ACCEPT
Candidate context: The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China

Question: When was the Yuan dynasty proclaimed?
Top similarity: 0.4806
Threshold decision: ACCEPT
Candidate context: The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China

Question: What is the capital of Mars?
Top similarity: 0.1721
Threshold decision: REJECT

Question: What is the University of Chicago?
Top similarity: 0.5186
Threshold decision: ACCEPT
Candidate context: The University of Chicago (UChicago, Chicago, or U of C) is a private research university in Chicago. The university, established in 1890, consists of The College, various graduate


In [8]:

from src.pipeline import answer_question
from src.model import build_messages

print("Pipeline import: PASSED")

messages = build_messages(
    context="The Yuan dynasty was established by Kublai Khan in 1271.",
    question="Who established the Yuan dynasty?"
)

print("System role:", messages[0]["role"])
print("User role:", messages[1]["role"])
print("\nPipeline and prompt components are connected.")


Pipeline import: PASSED
System role: system
User role: user

Pipeline and prompt components are connected.


In [9]:

print("Integration files:")
for filename in ["schema.py", "tools.py", "model.py", "pipeline.py"]:
    path = PROJECT_ROOT / "src" / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nNotebook:", Path.cwd())
print("Evaluation dataset:", data_path)
print("Unique retrieval contexts:", len(retriever.contexts))


Integration files:
schema.py: FOUND
tools.py: FOUND
model.py: FOUND
pipeline.py: FOUND

Notebook: c:\Users\HP\AppliedGenAIAssignments\notebooks
Evaluation dataset: c:\Users\HP\AppliedGenAIAssignments\data\eval_set.jsonl
Unique retrieval contexts: 35


In [10]:

import importlib
import src.pipeline

importlib.reload(src.pipeline)

print("Pipeline file:", src.pipeline.__file__)
print("Threshold defined:", hasattr(src.pipeline, "DEFAULT_SIMILARITY_THRESHOLD"))
print("Available threshold:", getattr(src.pipeline, "DEFAULT_SIMILARITY_THRESHOLD", "NOT FOUND"))


Pipeline file: c:\Users\HP\AppliedGenAIAssignments\src\pipeline.py
Threshold defined: True
Available threshold: 0.2


In [11]:

from src.pipeline import DEFAULT_SIMILARITY_THRESHOLD

print("Pipeline threshold:", DEFAULT_SIMILARITY_THRESHOLD)

# Simulate the pipeline's threshold decision without calling Qwen.
test_scores = {
    "Relevant Yuan dynasty passage": 0.4939,
    "Irrelevant Mars question": 0.1721,
}

for label, score in test_scores.items():
    decision = (
        "PASS TO MODEL"
        if score >= DEFAULT_SIMILARITY_THRESHOLD
        else "ABSTAIN"
    )
    print(f"{label}: score={score:.4f} -> {decision}")


Pipeline threshold: 0.2
Relevant Yuan dynasty passage: score=0.4939 -> PASS TO MODEL
Irrelevant Mars question: score=0.1721 -> ABSTAIN


In [12]:

from src.schema import validate_output, OutputValidationError

context = "The Yuan dynasty was established by Kublai Khan in 1271."

# Test 1: A valid answer
valid_answer = {
    "answerable": True,
    "answer": "Kublai Khan",
    "evidence": context,
    "confidence": 0.95
}

result = validate_output(valid_answer, context)
print("Valid answer test: PASSED")
print(result)

# Test 2: A valid abstention
valid_abstention = {
    "answerable": False,
    "answer": "",
    "evidence": "",
    "confidence": 0.90
}

result = validate_output(valid_abstention, context)
print("\nAbstention test: PASSED")
print(result)


Valid answer test: PASSED
{'answerable': True, 'answer': 'Kublai Khan', 'evidence': 'The Yuan dynasty was established by Kublai Khan in 1271.', 'confidence': 0.95}

Abstention test: PASSED
{'answerable': False, 'answer': '', 'evidence': '', 'confidence': 0.9}


In [13]:

from src.pipeline import answer_question

class FakeRetriever:
    def retrieve(self, question):
        return [{"context": "A passage about Warsaw.", "score": 0.1721}]

result = answer_question(
    question="What is the capital of Mars?",
    retriever=FakeRetriever(),
    tokenizer=None,
    model=None,
)

print(result)

assert result["answerable"] is False
assert result["answer"] == ""
assert result["evidence"] == ""

print("\nPipeline abstention test: PASSED")


{'answerable': False, 'answer': '', 'evidence': '', 'confidence': 0.0}

Pipeline abstention test: PASSED


In [14]:

import src.pipeline

class RelevantFakeRetriever:
    def retrieve(self, question):
        return [{
            "context": "The Yuan dynasty was established by Kublai Khan in 1271.",
            "score": 0.4939,
        }]

def fake_generate_answer(context, question, tokenizer, model):
    return {
        "answerable": True,
        "answer": "Kublai Khan",
        "evidence": context,
        "confidence": 0.95,
    }

# Replace generation temporarily for this test only.
original_generate_answer = src.pipeline.generate_answer
src.pipeline.generate_answer = fake_generate_answer

try:
    result = src.pipeline.answer_question(
        question="Who established the Yuan dynasty?",
        retriever=RelevantFakeRetriever(),
        tokenizer=None,
        model=None,
    )

    print(result)
    assert result["answerable"] is True
    assert result["answer"] == "Kublai Khan"
    print("\nRelevant-context pipeline test: PASSED")
finally:
    src.pipeline.generate_answer = original_generate_answer


{'answerable': True, 'answer': 'Kublai Khan', 'evidence': 'The Yuan dynasty was established by Kublai Khan in 1271.', 'confidence': 0.95}

Relevant-context pipeline test: PASSED


In [15]:

print("OM — STRUCTURED OUTPUT & TOOLS CHECKLIST")
print("----------------------------------------")
print("1. Schema validation: PASSED")
print("2. Valid abstention validation: PASSED")
print("3. TF-IDF context retrieval: PASSED")
print("4. Similarity threshold diagnostic: PASSED")
print("5. Unsupported evidence rejection: PASSED")
print("6. Pipeline abstention path: PASSED")
print("7. Pipeline answer path (mock generator): PASSED")
print("\nNote: Actual Qwen inference has not been tested in this environment.")


OM — STRUCTURED OUTPUT & TOOLS CHECKLIST
----------------------------------------
1. Schema validation: PASSED
2. Valid abstention validation: PASSED
3. TF-IDF context retrieval: PASSED
4. Similarity threshold diagnostic: PASSED
5. Unsupported evidence rejection: PASSED
6. Pipeline abstention path: PASSED
7. Pipeline answer path (mock generator): PASSED

Note: Actual Qwen inference has not been tested in this environment.


In [16]:
import sys
print(sys.version)
print(sys.executable)

3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
c:\Users\HP\AppliedGenAIAssignments\.venv312\Scripts\python.exe
